In [1]:
import os

os.makedirs(

    "k8s",

    exist_ok=True

)

print("✅ k8s folder created")

✅ k8s folder created


In [2]:
%%writefile k8s/deployment.yaml

apiVersion: apps/v1
kind: Deployment

metadata:
  name: retailpulse-app
  labels:
    app: retailpulse

spec:
  replicas: 2

  selector:
    matchLabels:
      app: retailpulse

  template:

    metadata:
      labels:
        app: retailpulse

    spec:

      containers:

      - name: retailpulse

        image: ghcr.io/yourname/retailpulse:latest

        ports:
        - containerPort: 8501

        resources:

          requests:
            cpu: "500m"
            memory: "1Gi"

          limits:
            cpu: "2"
            memory: "4Gi"

        livenessProbe:

          httpGet:
            path: /_stcore/health
            port: 8501

          initialDelaySeconds: 30

        readinessProbe:

          httpGet:
            path: /_stcore/health
            port: 8501

          initialDelaySeconds: 10

---

apiVersion: v1
kind: Service

metadata:
  name: retailpulse-svc

spec:

  type: LoadBalancer

  selector:
    app: retailpulse

  ports:

  - port: 80
    targetPort: 8501

---

apiVersion: autoscaling/v2
kind: HorizontalPodAutoscaler

metadata:
  name: retailpulse-hpa

spec:

  scaleTargetRef:

    apiVersion: apps/v1
    kind: Deployment
    name: retailpulse-app

  minReplicas: 2

  maxReplicas: 8

  metrics:

  - type: Resource

    resource:

      name: cpu

      target:

        type: Utilization

        averageUtilization: 70

Writing k8s/deployment.yaml
